# MLP FINAL
- Renskrevet version af 'mlp-v1'.
- Data klagøres i '00_data-prep'

- I denne notebook er der derfor kun:
- Encoding af features
- Afsøgning af hyperparametre
- Retraining på den bedste config
- Evaluering på test data

## 1. Setup

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn import preprocessing
from tensorflow import keras

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
keras.utils.set_random_seed(SEED)

DATA_DIR = Path('../data')
ARTIFACTS_DIR = Path('artifacts')
ARTIFACTS_DIR.mkdir(exist_ok=True)

BATCH_SIZE = 32
MAX_EPOCHS = 30
PATIENCE = 5

## 2. Load Data

- Datasættet er allerede splittet (64/16/20 - train/val/test) og stratified på target.
- Her loader vi data ind og verificerer shape er som forventet.

In [2]:
X_train = pd.read_csv(DATA_DIR / 'X_train.csv')
X_val = pd.read_csv(DATA_DIR / 'X_val.csv')
X_test = pd.read_csv(DATA_DIR / 'X_test.csv')

y_train = pd.read_csv(DATA_DIR / 'y_train.csv').squeeze('columns')
y_val = pd.read_csv(DATA_DIR / 'y_val.csv').squeeze('columns')
y_test = pd.read_csv(DATA_DIR / 'y_test.csv').squeeze('columns')

y_train = (y_train == '>50K').astype(int)
y_val   = (y_val   == '>50K').astype(int)
y_test  = (y_test  == '>50K').astype(int)

# Positive rate er antallet af samples hvor y == 1 som er: >50K
# Hvis stratify har virket, skal fordelingen af targets være ca. det samme i hvert sæt.
print(f"train: {X_train.shape}      positive rate: {y_train.mean():.3f}")
print(f"val:   {X_val.shape}       positive rate: {y_val.mean():.3f}")
print(f"test:  {X_test.shape}       positive rate: {y_test.mean():.3f}")

train: (28941, 13)      positive rate: 0.248
val:   (7236, 13)       positive rate: 0.248
test:  (9045, 13)       positive rate: 0.248


## 3. Feature Encoding
- Numeriske features > StandardScalar
- Kategoriske features > OneHotEncoder
- Transformeren fittes kun på train, og bruges derefter til at tranformere val og test.
- Ellers leaker statistik fra val og test ind i train.

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

num_cols = ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
cat_cols = ['workclass', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country']

preprocessor = ColumnTransformer([
      ('num', StandardScaler(),                          num_cols),
      ('cat', OneHotEncoder(handle_unknown='ignore'),    cat_cols),
  ])

X_train_enc = preprocessor.fit_transform(X_train)
X_val_enc   = preprocessor.transform(X_val)
X_test_enc  = preprocessor.transform(X_test)

